In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from matplotlib.ticker import MultipleLocator, FuncFormatter
from comet_ml.query import Parameter, Tag
import comet_ml

# ==========================================
# USER CONFIGURATION
# ==========================================
PROJECT_NAME = "segdac"
WORKSPACE_NAME = "alexandrebrown"

# LIST OF TASKS TO PROCESS
TASKS_TO_PLOT = [
    "PickCube-v1",
    "PullCubeTool-v1",
    "PushCube-v1"
]

# Tag to identify the ablation runs generally
ABLATION_TAG = "text_tags_ablation"
BASELINE_TAG = "paper-result"

# Smoothing factor (0.9 is standard for N=3-5 seeds)
SMOOTHING_ALPHA = 0.9 
MIN_STEP = 10_000

# ==========================================
# MAPPINGS & SETUP
# ==========================================
logged_task_names = {
    "PushCube-v1": ["PushCube-v1", "PushCubeFixed-v1"],
    "PickCube-v1": ["PickCube-v1", "PickCubeVisual-v1"],
    "PullCubeTool-v1": ["PullCubeTool-v1"],
}

color_map = {
    "Paper Result": "#000000",      # Black
    "Text Ablation 1": "#D55E00",   # Vermilion/Orange
    "Text Ablation 2": "#0072B2",   # Blue
}

output_plots_folder = Path("results/ablation_plots")
output_plots_folder.mkdir(parents=True, exist_ok=True)
sns.set_theme(context="paper", style="whitegrid", font_scale=1.2)

# Initialize API once
api = comet_ml.API()

# ==========================================
# HELPER FUNCTIONS
# ==========================================
def xfmt(x, pos):
    if x >= 1_000_000:
        return f"{int(x/1_000_000)}M"
    else:
        return f"{int(x/1000)}k"

def smooth_curve(curve, alpha):
    """
    Standard Exponential Moving Average.
    Now applied to the AGGREGATED mean, not individual seeds.
    """
    if alpha == 0:
        return curve
    beta = alpha
    alpha = 1 - beta
    smoothed = np.zeros_like(curve)
    smoothed[0] = curve[0]
    for i in range(1, len(curve)):
        smoothed[i] = alpha * curve[i] + beta * smoothed[i - 1]
    return smoothed

def get_raw_aggregated_data(experiments, label):
    """
    1. Downloads raw return curves.
    2. Does NOT smooth individual seeds.
    3. Aggregates (Mean +/- Std) on raw data.
    """
    seed_step_to_ret = []
    
    print(f"    Processing {len(experiments)} runs for '{label}'...")

    for exp in experiments:
        # Get raw metrics
        try:
            raw_metrics = exp.get_metrics("eval_return")
        except Exception as e:
            print(f"      [Err] Failed to get metrics for {exp.get_name()}: {e}")
            continue

        if not raw_metrics:
            continue
            
        experiment_returns = sorted(raw_metrics, key=lambda x: x["step"])
        
        # Deduplicate steps
        unique_returns = {}
        for r in experiment_returns:
            unique_returns[int(r["step"])] = float(r["metricValue"])

        # Filter MIN_STEP
        filtered_returns = {s: v for s, v in unique_returns.items() if s >= MIN_STEP}

        if filtered_returns:
            # 1. Sort Data
            seed_steps = sorted(filtered_returns.keys())
            # 2. Store RAW values (No smoothing here)
            seed_vals_raw = np.array([filtered_returns[s] for s in seed_steps], dtype=float)
            
            # 3. Store
            seed_step_to_ret.append(dict(zip(seed_steps, seed_vals_raw)))

    if not seed_step_to_ret:
        print(f"      [WARN] No valid data found for {label}")
        return None, None, None

    # Aggregate
    all_steps = sorted({s for d in seed_step_to_ret for s in d.keys()})
    all_steps_arr = np.array(all_steps, dtype=int)
    
    means = []
    stds = []
    
    for s in all_steps:
        # Get values from all seeds that have this step
        vals = [d[s] for d in seed_step_to_ret if s in d]
        vals = np.asarray(vals, dtype=float)
        means.append(vals.mean())
        stds.append(vals.std())
        
    return all_steps_arr, np.array(means), np.array(stds)

# ==========================================
# CORE PLOTTING FUNCTION
# ==========================================
def generate_ablation_plot(task_key):
    print(f"\n=== Generating Plot for Task: {task_key} ===")

    # 1. Identify Env IDs
    target_env_ids = logged_task_names.get(task_key, [task_key])

    # 2. Fetch ALL Ablation Experiments (Tag based)
    all_ablation_experiments = []
    for env_id in target_env_ids:
        query = (Parameter("env|id") == env_id) & Tag(ABLATION_TAG)
        all_ablation_experiments += api.query(WORKSPACE_NAME, PROJECT_NAME, query)

    if not all_ablation_experiments:
        print(f"  [ERROR] No experiments found with tag '{ABLATION_TAG}' for task {task_key}")
        return

    # 3. Separate Ablation 1 and Ablation 2 based on NAME
    ablation_1_exps = []
    ablation_2_exps = []
    target_seeds = set()

    print(f"  Filtering {len(all_ablation_experiments)} ablation runs by name...")

    for exp in all_ablation_experiments:
        name = exp.get_name()
        
        # Store seed for baseline matching
        try:
            s_val = exp.get_parameters_summary("evaluation|seed")["valueCurrent"]
            target_seeds.add(s_val)
        except:
            pass

        # Logic to split by name
        if "ablation_1" in name or name.endswith("_1"):
            ablation_1_exps.append(exp)
        elif "ablation_2" in name or name.endswith("_2"):
            ablation_2_exps.append(exp)
        else:
            pass 

    print(f"  -> Found {len(ablation_1_exps)} runs for Ablation 1")
    print(f"  -> Found {len(ablation_2_exps)} runs for Ablation 2")

    # 4. Fetch Matching Paper Results (Same Task + Same Seeds)
    paper_experiments = []
    for env_id in target_env_ids:
        query = (Parameter("env|id") == env_id) & Tag(BASELINE_TAG)
        candidates = api.query(WORKSPACE_NAME, PROJECT_NAME, query)
        
        # Filter locally for matching seeds
        for exp in candidates:
            try:
                s_val = exp.get_parameters_summary("evaluation|seed")["valueCurrent"]
                if s_val in target_seeds:
                    paper_experiments.append(exp)
            except:
                continue

    print(f"  -> Found {len(paper_experiments)} matching 'paper-result' runs.")

    # 5. PLOTTING
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # We use a list to track max step for plotting limits
    max_steps_found = [MIN_STEP]

    # Inner helper to plot one line
    def plot_line(exps, label, color):
        # 1. Get RAW Aggregated Data
        steps, raw_means, raw_stds = get_raw_aggregated_data(exps, label)
        
        if steps is not None and len(steps) > 0:
            # 2. Smooth ONLY the Mean Curve
            smoothed_means = smooth_curve(raw_means, SMOOTHING_ALPHA)
            
            # 3. Plot smoothed mean
            ax.plot(steps, smoothed_means, label=label, color=color, linewidth=2.5)
            
            # 4. Fill using Smoothed Mean +/- Raw Std
            # This centers the variance around the clean line, but keeps true variance
            lower = smoothed_means - raw_stds
            upper = smoothed_means + raw_stds
            ax.fill_between(steps, lower, upper, color=color, alpha=0.15)
            
            max_steps_found.append(steps[-1])

    # Plot the 3 lines
    plot_line(paper_experiments, "Paper Result", color_map["Paper Result"])
    plot_line(ablation_1_exps, "Text Inputs Synonyms 1", color_map["Text Ablation 1"])
    plot_line(ablation_2_exps, "Text Inputs Synonyms 2", color_map["Text Ablation 2"])

    # Formatting
    ax.set_title(f"Text Inputs Synonyms Ablation ({task_key})", fontsize=16, fontweight='bold', pad=12)
    ax.set_ylabel("Eval Return", fontsize=14)
    ax.set_xlabel("Steps", fontsize=14)

    # Smart X-Axis Formatting
    last_step = max(max_steps_found)
    ax.xaxis.set_major_formatter(FuncFormatter(xfmt))
    ax.set_xlim(MIN_STEP, last_step)
    ax.grid(True, which="major", axis="y", linestyle="--", alpha=0.5)

    # Legend
    ax.legend(fontsize=12, loc="lower right", frameon=True)

    sns.despine()
    plt.tight_layout()

    filename = output_plots_folder / f"{task_key.lower()}_synonyms_ablation.pdf"
    fig.savefig(filename)
    print(f"  Plot saved to: {filename}")
    
    # Close figure to free memory
    plt.close(fig) 


print(f"Starting batch generation for {len(TASKS_TO_PLOT)} tasks...")

for task_name in TASKS_TO_PLOT:
    try:
        generate_ablation_plot(task_name)
    except Exception as e:
        print(f"CRITICAL ERROR processing {task_name}: {e}")
        import traceback
        traceback.print_exc()

    print("\nAll tasks processed.")

Starting batch generation for 3 tasks...

=== Generating Plot for Task: PickCube-v1 ===
  Filtering 6 ablation runs by name...
  -> Found 3 runs for Ablation 1
  -> Found 3 runs for Ablation 2
  -> Found 3 matching 'paper-result' runs.
    Processing 3 runs for 'Paper Result'...
    Processing 3 runs for 'Text Inputs Synonyms 1'...
    Processing 3 runs for 'Text Inputs Synonyms 2'...
  Plot saved to: results/ablation_plots/pickcube-v1_synonyms_ablation.pdf

All tasks processed.

=== Generating Plot for Task: PullCubeTool-v1 ===
  Filtering 6 ablation runs by name...
  -> Found 3 runs for Ablation 1
  -> Found 3 runs for Ablation 2
  -> Found 3 matching 'paper-result' runs.
    Processing 3 runs for 'Paper Result'...
    Processing 3 runs for 'Text Inputs Synonyms 1'...
    Processing 3 runs for 'Text Inputs Synonyms 2'...
  Plot saved to: results/ablation_plots/pullcubetool-v1_synonyms_ablation.pdf

All tasks processed.

=== Generating Plot for Task: PushCube-v1 ===
  Filtering 6 abla